# The Rotation That Turned Out To Be a Mirror

**Topic:** Orthogonal Procrustes &nbsp;·&nbsp; **Difficulty:** Hard

Aligning two point clouds is a three-line SVD recipe: centre both, take the SVD
of the cross-covariance, multiply the singular vectors. The three-line version is
also wrong about 50% of the time, because the orthogonal matrix it returns may
have determinant `-1`: a reflection, not a rotation.

For point clouds that is often fatal: a mirrored molecule is a different
molecule. Implement both, and fix the sign.

Prerequisite: an orthogonal matrix has determinant `+1` or `-1`, and the sign tells a
rotation from a reflection. Linear Algebra I covers this in its optional section
"Orthogonal matrices and Gram-Schmidt".

## The data (created in the next cell)

- `A` `(150, 3)`: a reference point cloud.
- `B` `(150, 3)`: the same cloud rotated, translated, and lightly noised.
- `B_mirror`: the same but with a genuine reflection applied.

## Your task

With **NumPy only**:

1. Implement `procrustes(P, Q, allow_reflection)`: centre both clouds, form
   `H = Pc.T @ Qc`, take `U, S, Vt = svd(H)`, and set `R = V @ U.T`.
   When `allow_reflection` is `False`, flip the sign of the last column of `V`
   if `det(V @ U.T) < 0` before forming `R`.
2. Align `B` to `A` and `B_mirror` to `A`, both ways.
3. Set `answers` to a dict with `"rmse_B_free"`, `"rmse_B_proper"`,
   `"rmse_mirror_free"`, `"rmse_mirror_proper"` (the RMSE between the aligned
   cloud and the cloud it was aligned to, `B` for the first two and `B_mirror`
   for the last two, in each of the four cases) and `"det_free_mirror"`, the
   determinant of the unconstrained `R` fitted to `B_mirror`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the unconstrained fit to the mirrored cloud has determinant `-1` and that
the proper-rotation fit cannot match it.

In [ ]:
import numpy as np

rng = np.random.default_rng(3002)
A = rng.normal(0, 1, (150, 3))

theta = 0.9
Rz = np.array([[np.cos(theta), -np.sin(theta), 0],
               [np.sin(theta), np.cos(theta), 0],
               [0, 0, 1.0]])
t = np.array([2.0, -1.0, 0.5])
B = A @ Rz.T + t + rng.normal(0, 0.01, (150, 3))

Mrf = np.diag([1.0, 1.0, -1.0])
B_mirror = A @ Mrf @ Rz.T + t + rng.normal(0, 0.01, (150, 3))
print(f"A {A.shape}. B is a rotation of A, and B_mirror includes a reflection")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("rmse_B_free", [], 0.01719628457271514, -0.0028793117510406206, 1e-06),
    ("rmse_B_proper", [], 0.01719628457271514, -0.0028793117510406206, 1e-06),
    ("rmse_mirror_free", [], 0.01689189822833313, -0.0028283459058006544, 1e-06),
    ("rmse_mirror_proper", [], 1.8195254585448624, -0.304657730683193, 1e-06),
    ("det_free_mirror", [], -1.0000000000000009, 0.1674380148144993, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "det_free_mirror" in answers:
    _neg = abs(answers["det_free_mirror"] + 1.0) < 1e-6
    _gap = answers["rmse_mirror_proper"] > 10 * answers["rmse_mirror_free"]
    print(f"  {'free fit is a reflection':<24} {'ok' if _neg else 'wrong'}  "
          f"(det = {answers['det_free_mirror']:+.4f})")
    print(f"  {'rotation cannot match it':<24} {'ok' if _gap else 'wrong'}")
    _ok = _ok and _neg and _gap

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

`V @ U.T` is the closest *orthogonal* matrix, and the orthogonal group has two
components: rotations with determinant `+1` and reflections with `-1`. Flipping
the sign of the last column of `V` (the one paired with the smallest singular
value) moves you to the rotation component at the least possible cost. The
mirrored cloud is the diagnostic: an unconstrained fit matches it almost exactly
with `det = -1`, and no proper rotation can, which is exactly the information you
want when chirality matters.

```python
import numpy as np

def procrustes(P, Q, allow_reflection):
    cp, cq = P.mean(axis=0), Q.mean(axis=0)
    Pc, Qc = P - cp, Q - cq
    H = Pc.T @ Qc
    U, S, Vt = np.linalg.svd(H)
    V = Vt.T
    R = V @ U.T
    if not allow_reflection and np.linalg.det(R) < 0:
        V = V.copy()
        V[:, -1] *= -1
        R = V @ U.T
    aligned = Pc @ R.T + cq
    return R, aligned

def rmse(x, y):
    return float(np.sqrt(np.mean(np.sum((x - y) ** 2, axis=1))))

_, aB_free = procrustes(A, B, True)
_, aB_prop = procrustes(A, B, False)
Rm_free, aM_free = procrustes(A, B_mirror, True)
_, aM_prop = procrustes(A, B_mirror, False)

answers = {
    "rmse_B_free": rmse(aB_free, B),
    "rmse_B_proper": rmse(aB_prop, B),
    "rmse_mirror_free": rmse(aM_free, B_mirror),
    "rmse_mirror_proper": rmse(aM_prop, B_mirror),
    "det_free_mirror": float(np.linalg.det(Rm_free)),
}
```